# 03 — Model 1: Machine Failure Prediction

**Goal:** predict whether the machine will fail from its type and five sensor readings.

The model is trained on the training split, the alert threshold is selected using validation data, and the test set is used only for final evaluation.

In [4]:
import os
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = "../data/predictive_maintenance.csv"
MODEL_DIR = "../models"
OUT_DIR = "../outputs/model_failure"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
feature_cols = ["Type", "Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
X = df[feature_cols].copy()
y = df["Machine failure"].copy()

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.20, stratify=y_temp, random_state=42)

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["Type"]),
    ("num", StandardScaler(), ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"])
])

X_train_p = preprocessor.fit_transform(X_train)
X_val_p = preprocessor.transform(X_val)
X_test_p = preprocessor.transform(X_test)


In [5]:
model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
model.fit(X_train_p, y_train)
val_prob = model.predict_proba(X_val_p)[:, 1]
print("Random Forest trained successfully.")


Random Forest trained successfully.


## Threshold selection

We require recall of at least 70% on validation data, then choose the threshold with the highest precision among those candidates.

In [6]:
import numpy as np

thresholds = np.arange(0.10, 0.61, 0.05)
rows = []
for threshold in thresholds:
    pred = (val_prob >= threshold).astype(int)
    rows.append({
        "threshold": float(threshold),
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0)
    })

threshold_results = pd.DataFrame(rows)
valid = threshold_results[threshold_results["recall"] >= 0.70]
best_threshold = float(valid.loc[valid["precision"].idxmax(), "threshold"])
print(threshold_results.round(4))
print("\nSelected threshold:", best_threshold)


    threshold  precision  recall      f1
0        0.10     0.2400  0.8889  0.3780
1        0.15     0.2956  0.8704  0.4413
2        0.20     0.3511  0.8519  0.4973
3        0.25     0.3929  0.8148  0.5301
4        0.30     0.4505  0.7593  0.5655
5        0.35     0.5063  0.7407  0.6015
6        0.40     0.5373  0.6667  0.5950
7        0.45     0.5833  0.6481  0.6140
8        0.50     0.6939  0.6296  0.6602
9        0.55     0.7619  0.5926  0.6667
10       0.60     0.8108  0.5556  0.6593

Selected threshold: 0.3500000000000001


In [7]:
test_prob = model.predict_proba(X_test_p)[:, 1]
test_pred = (test_prob >= best_threshold).astype(int)

print("=== Final Test Report ===")
print(classification_report(y_test, test_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, test_pred))
print("ROC-AUC:", round(roc_auc_score(y_test, test_prob), 4))
print("PR-AUC:", round(average_precision_score(y_test, test_prob), 4))


=== Final Test Report ===
              precision    recall  f1-score   support

           0     0.9921    0.9736    0.9828      1932
           1     0.5096    0.7794    0.6163        68

    accuracy                         0.9670      2000
   macro avg     0.7509    0.8765    0.7995      2000
weighted avg     0.9757    0.9670    0.9703      2000

Confusion matrix:
 [[1881   51]
 [  15   53]]
ROC-AUC: 0.9594
PR-AUC: 0.7137


In [8]:
# Save the trained model and preprocessing pipeline
joblib.dump(model, os.path.join(MODEL_DIR, "model_failure_rf.joblib"))
joblib.dump(preprocessor, os.path.join(MODEL_DIR, "model_failure_preprocessor.joblib"))
joblib.dump({"threshold": best_threshold, "feature_cols": feature_cols}, os.path.join(MODEL_DIR, "model_failure_metadata.joblib"))

metrics = {
    "threshold": best_threshold,
    "failure_precision": precision_score(y_test, test_pred, zero_division=0),
    "failure_recall": recall_score(y_test, test_pred, zero_division=0),
    "failure_f1": f1_score(y_test, test_pred, zero_division=0),
    "accuracy": float((test_pred == y_test).mean()),
    "roc_auc": roc_auc_score(y_test, test_prob),
    "pr_auc": average_precision_score(y_test, test_prob)
}
pd.Series(metrics).to_json(os.path.join(OUT_DIR, "metrics.json"), indent=2)

import json

with open(os.path.join(OUT_DIR, "confusion_matrix.json"), "w") as f:
    json.dump(confusion_matrix(y_test, test_pred).tolist(), f, indent=2)
print("Saved Model 1 artifacts.")


Saved Model 1 artifacts.
